# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will inspect all available record sets in this dataset, listing their `@id`s and the fields they contain.

In [ ]:
# List available record sets by their @ids

record_sets = dataset.metadata.record_sets

if not record_sets:
    print('No record sets found in the metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"    - {field.name} (id: {field.id}, type: {field.data_type})")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Find record sets and their IDs
record_sets = dataset.metadata.record_sets
dataframes = {}

if not record_sets:
    print('No record sets available for data extraction.')
else:
    for rs in record_sets:
        rs_id = rs.id
        print(f"Loading records for RecordSet: {rs.name} | id: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(df.head())
        except Exception as e:
            print(f"  ERROR loading records: {e}")
    # Pick one for downstream analysis
    chosen_rs = list(dataframes.keys())[0] if dataframes else None
    if chosen_rs:
        print(f"Chosen RecordSet for further analysis: {chosen_rs}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's pick a numeric field from our chosen RecordSet (update the variables below as appropriate based on your fields).

In [ ]:
# NOTE: Set these to the actual field @id values shown above for your chosen record set.
record_set_id = chosen_rs

df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# Print names to assist users in selecting fields:
print(f"DataFrame columns: {df.columns.tolist()}")

# Example: Let's attempt to select a numeric column automatically (user can override)
numeric_fields = [c for c in df.columns if df[c].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[c])]
print(f"Numeric fields: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]  # Use the first found numeric field

    threshold = df[numeric_field].mean() if df[numeric_field].mean() is not None else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by the first non-numeric column
    group_fields = [c for c in df.columns if c != numeric_field and not pd.api.types.is_numeric_dtype(df[c])]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
else:
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped, make a boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Not enough numeric data to plot.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We used `mlcroissant` to load the FAIR² mass spectrometry dataset.
- The dataset structure, available record sets, and fields were reviewed using their `@id`s.
- A numeric field was selected for statistical analysis, normalization, and group-wise comparisons (if applicable).
- Data distributions were visualized to aid interpretation.

**You can now proceed with further domain-specific protein analysis!**